Test of the API requests

In [61]:
import sys
from pathlib import Path
project_root = Path.cwd().parent
sys.path.insert(0,str(project_root))
print(sys.path[:3])
import src
print(src)
print(src.__file__)

['/Users/nathanchallande/Documents/ETHZ/portfolio/air-quality-prediction', '/Users/nathanchallande/Documents/ETHZ/portfolio/air-quality-prediction', '/Users/nathanchallande/Documents/ETHZ/portfolio/air-quality-prediction']
<module 'src' from '/Users/nathanchallande/Documents/ETHZ/portfolio/air-quality-prediction/src/__init__.py'>
/Users/nathanchallande/Documents/ETHZ/portfolio/air-quality-prediction/src/__init__.py


In [62]:
from src.api import get_weather

latitude = 39.982
longitude = 116.397

weather = get_weather(
    latitude,
    longitude
)

weather
from src.api import get_air_quality

air_quality = get_air_quality(
    latitude,
    longitude
)

print(air_quality["current"])

{'time': '2026-08-16T18:00', 'interval': 3600, 'pm10': 91.3, 'carbon_monoxide': 544.0, 'nitrogen_dioxide': 64.4, 'sulphur_dioxide': 4.5, 'ozone': 75.0}


In [63]:
from src.api import get_weather, get_air_quality, create_api_features
features = create_api_features(weather,air_quality)
print(features)

{'TEMP': 25.4, 'PRES': 999.8, 'DEWP': 19.3, 'RAIN': 0.0, 'WSPM': 0.32, 'wd': 'NNE', 'PM10': 91.3, 'CO': 544.0, 'NO2': 64.4, 'SO2': 4.5, 'O3': 75.0}


In [64]:
from src.api import create_api_dataframe
api_df = create_api_dataframe(
    features,
    air_quality["current"]["time"],
    "Aotizhongxin"
)
print(api_df.columns.tolist())

['TEMP', 'PRES', 'DEWP', 'RAIN', 'WSPM', 'wd', 'PM10', 'CO', 'NO2', 'SO2', 'O3', 'datetime', 'station']


The connection with the API are well established.
Now we will assume we are working with data similar to the station Aotizhongxin.

In [65]:
from src.api import add_api_datetime_features
station = "Aotizhongxin"
#the time stamp isn't the same for the weather and the air-quality prediction. Therefor we choose the one with hourly observations(air-quality)
timestamp = air_quality["current"]["time"]
print(timestamp)
features = add_api_datetime_features(
    features,
    air_quality["current"]["time"]
)

print(features)

2026-08-16T18:00
{'TEMP': 25.4, 'PRES': 999.8, 'DEWP': 19.3, 'RAIN': 0.0, 'WSPM': 0.32, 'wd': 'NNE', 'PM10': 91.3, 'CO': 544.0, 'NO2': 64.4, 'SO2': 4.5, 'O3': 75.0, 'year': 2026, 'month': 8, 'day': 16, 'hour': 18, 'dayofweek': 6}


In [66]:
#then we convert the temporal values into the cosin and sin format
import pandas as pd
from src.api import add_cyclic_features
api_df = add_cyclic_features(api_df)
print(api_df.columns.tolist())

['TEMP', 'PRES', 'DEWP', 'RAIN', 'WSPM', 'wd', 'PM10', 'CO', 'NO2', 'SO2', 'O3', 'datetime', 'station', 'hour', 'month', 'day_of_week', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'day_of_week_sin', 'day_of_week_cos']


Now we use the encoder from the preprocessing file for the inference features

In [67]:
from src.preprocessing import encoder
print(encoder.categories_)
print(encoder.get_feature_names_out(["wd","station"]))

[array(['E', 'ENE', 'ESE', 'N', 'NE', 'NNE', 'NNW', 'NW', 'S', 'SE', 'SSE',
       'SSW', 'SW', 'W', 'WNW', 'WSW'], dtype=object), array(['Aotizhongxin', 'Changping', 'Dingling', 'Dongsi', 'Guanyuan',
       'Gucheng', 'Huairou', 'Nongzhanguan', 'Shunyi', 'Tiantan',
       'Wanliu', 'Wanshouxigong'], dtype=object)]
['wd_E' 'wd_ENE' 'wd_ESE' 'wd_N' 'wd_NE' 'wd_NNE' 'wd_NNW' 'wd_NW' 'wd_S'
 'wd_SE' 'wd_SSE' 'wd_SSW' 'wd_SW' 'wd_W' 'wd_WNW' 'wd_WSW'
 'station_Aotizhongxin' 'station_Changping' 'station_Dingling'
 'station_Dongsi' 'station_Guanyuan' 'station_Gucheng' 'station_Huairou'
 'station_Nongzhanguan' 'station_Shunyi' 'station_Tiantan'
 'station_Wanliu' 'station_Wanshouxigong']


In [68]:
api_categorical = encoder.transform(api_df[["wd","station"]])
print(api_categorical.shape)
print(api_categorical)

(1, 28)
[[0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0. 0.]]


In [69]:
api_categorical_df = pd.DataFrame(
    api_categorical,
    columns=encoder.get_feature_names_out(
        ["wd", "station"]
    )
)

print(api_categorical_df.T)

                         0
wd_E                   0.0
wd_ENE                 0.0
wd_ESE                 0.0
wd_N                   0.0
wd_NE                  0.0
wd_NNE                 1.0
wd_NNW                 0.0
wd_NW                  0.0
wd_S                   0.0
wd_SE                  0.0
wd_SSE                 0.0
wd_SSW                 0.0
wd_SW                  0.0
wd_W                   0.0
wd_WNW                 0.0
wd_WSW                 0.0
station_Aotizhongxin   1.0
station_Changping      0.0
station_Dingling       0.0
station_Dongsi         0.0
station_Guanyuan       0.0
station_Gucheng        0.0
station_Huairou        0.0
station_Nongzhanguan   0.0
station_Shunyi         0.0
station_Tiantan        0.0
station_Wanliu         0.0
station_Wanshouxigong  0.0


Now we define the numerical features used by the XGBoost model

In [70]:
numerical_features_pollution = [
    "TEMP",
    "PRES",
    "DEWP",
    "RAIN",
    "WSPM",
    "hour_sin",
    "hour_cos",
    "month_sin",
    "month_cos",
    "day_of_week_sin",
    "day_of_week_cos",
    "PM10",
    "SO2",
    "NO2",
    "CO",
    "O3"
]
api_numerical = api_df[numerical_features_pollution]
print(api_numerical)

   TEMP   PRES  DEWP  RAIN  WSPM  hour_sin      hour_cos  month_sin  \
0  25.4  999.8  19.3   0.0  0.32      -1.0 -1.836970e-16  -0.866025   

   month_cos  day_of_week_sin  day_of_week_cos  PM10  SO2   NO2     CO    O3  
0       -0.5        -0.781831          0.62349  91.3  4.5  64.4  544.0  75.0  


Now we combine the numerical and categorical features

In [71]:
X_api = pd.concat(
    [
        api_numerical.reset_index(drop=True),
        api_categorical_df.reset_index(drop=True)
    ],
    axis=1
)
print(X_api.shape)

(1, 44)


One important sanity check to do: Check if the features order of the X_api is the same as the X_train_pollution from the development part

In [72]:
from src.preprocessing import X_train_pollution
print(
    "Training shape:", X_train_pollution.shape)
print("API shape:", X_api.shape)

print("Number of expected features:", X_api.shape[1])
feature_names = (
    numerical_features_pollution
    + list(
        encoder.get_feature_names_out(
            ["wd", "station"]
        )
    )
)
print(
    "API columns correct:",
    list(X_api.columns) == feature_names
)

Training shape: (268076, 44)
API shape: (1, 44)
Number of expected features: 44
API columns correct: True


In [73]:
#We now have the correct features in the good order.
X_api = X_api[feature_names].to_numpy()
print(type(X_api))
print(X_api.shape)


<class 'numpy.ndarray'>
(1, 44)


Now we load the final XGBoost model

In [74]:
from xgboost import XGBRegressor
from pathlib import Path
loaded_model = XGBRegressor()
project_root = Path.cwd().parent
print(project_root)
loaded_model.load_model(project_root/"models/saved/xgboost_pollution.json")

/Users/nathanchallande/Documents/ETHZ/portfolio/air-quality-prediction


Now we can finally make the current predicition of PM2.5

In [75]:
prediction = loaded_model.predict(X_api)
#and we add our physical constraint by taking the max(0,y)
prediction = max(0.0,prediction[0])
print("Predicted PM2.5:", prediction)

Predicted PM2.5: 35.61432


In [76]:
print("=" * 40)
print("       AIR QUALITY PREDICTION")
print("=" * 40)

print(f"Station: Aotizhongxin")
print(f"Time: {air_quality['current']['time']}")

print()
print("Weather")
print("-" * 20)
print(f"Temperature: {features['TEMP']} °C")
print(f"Pressure:    {features['PRES']} hPa")
print(f"Dew point:   {features['DEWP']} °C")
print(f"Rain:        {features['RAIN']} mm")
print(f"Wind speed:  {features['WSPM']} m/s")
print(f"Wind:        {features['wd']}")

print()
print("Pollutants")
print("-" * 20)
print(f"PM10: {features['PM10']} µg/m³")
print(f"CO:   {features['CO']} µg/m³")
print(f"NO2:  {features['NO2']} µg/m³")
print(f"SO2:  {features['SO2']} µg/m³")
print(f"O3:   {features['O3']} µg/m³")

print()
print("-" * 40)
print(f"Predicted PM2.5: {prediction:.2f} µg/m³")
print("-" * 40)

       AIR QUALITY PREDICTION
Station: Aotizhongxin
Time: 2026-08-16T18:00

Weather
--------------------
Temperature: 25.4 °C
Pressure:    999.8 hPa
Dew point:   19.3 °C
Rain:        0.0 mm
Wind speed:  0.32 m/s
Wind:        NNE

Pollutants
--------------------
PM10: 91.3 µg/m³
CO:   544.0 µg/m³
NO2:  64.4 µg/m³
SO2:  4.5 µg/m³
O3:   75.0 µg/m³

----------------------------------------
Predicted PM2.5: 35.61 µg/m³
----------------------------------------


Now we test the prediction Using the global prediction function

In [77]:
from src.api import predict_current_pm25, load_model_and_encoder
model,encoder = load_model_and_encoder()
prediction = predict_current_pm25(
    station="Aotizhongxin",
    model=model,
    encoder=encoder
)
print("Predicted PM2.5:", prediction)

Predicted PM2.5: {'prediction': 35.61, 'timestamp': '2026-08-16T18:00', 'latitude': 39.982, 'longitude': 116.397, 'inputs': {'temperature': 25.4, 'pressure': 999.8, 'dew_point': 19.3, 'rain': 0.0, 'wind_speed': 0.32, 'wind_direction': 'NNE', 'pm10': 91.3, 'co': 544.0, 'no2': 64.4, 'so2': 4.5, 'o3': 75.0}}


We test the api requests for every station:

In [78]:
import requests

BASE_URL = "http://127.0.0.1:8000"

stations = [
    "Aotizhongxin",
    "Changping",
    "Dingling",
    "Dongsi",
    "Guanyuan",
    "Gucheng",
    "Huairou",
    "Nongzhanguan",
    "Shunyi",
    "Tiantan",
    "Wanliu",
    "Wanshouxigong",
]

for station in stations:

    response = requests.get(
        f"{BASE_URL}/predict",
        params={"station": station}
    )

    print(
        station,
        response.status_code,
        response.json()["predicted_pm25"]
    )
    

Aotizhongxin 200 55.45
Changping 200 41.51
Dingling 200 44.38
Dongsi 200 64.17
Guanyuan 200 61.21
Gucheng 200 29.08
Huairou 200 27.56
Nongzhanguan 200 59.93
Shunyi 200 36.14
Tiantan 200 67.22
Wanliu 200 64.24
Wanshouxigong 200 61.38
